In [1]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge

In [2]:
def interpolate_curve_by_x(curve):
    """
    Interpolates a 3D curve by x-coordinate (samplede at integer values) and returns (x, y, z),
    keeping the original start and end points exactly.

    Args:
        curve : array-like of shape (N, 3)
            Each row is [x, y, z].

    Returns:
        result : ndarray of shape (M, 3)
            Interpolated coordinates at integer x plus original endpoints.
    """

    curve = np.asarray(curve, dtype=float)
    if curve.ndim != 2 or curve.shape[1] != 3:
        raise ValueError("Data has the wrong shape")

    x = curve[:, 0]
    y = curve[:, 1]
    z = curve[:, 2]    

    x_new = np.arange(np.ceil(x.min()) + 0, np.floor(x.max()) + 1)

    #def f_y(x): values = interp1d(x, y, kind="linear", bounds_error=False, fill_value="extrapolate") return values
    #def f_z(x): values = interp1d(x, z, kind="linear", bounds_error=False, fill_value="extrapolate") return values
    
    f_y =  interp1d(x, y, kind="linear", bounds_error=False, fill_value="extrapolate")
    f_z = interp1d(x, z, kind="linear", bounds_error=False, fill_value="extrapolate")

    if len(x_new) > 0 :# or x[-1] != x_new[-1]:
        y_new = f_y(x_new)
        z_new = f_z(x_new)
        result = np.vstack([curve[0], np.column_stack((x_new, y_new, z_new)), curve[-1]])
    else:
        result = curve

    return result

In [3]:
def error_fun(curveA, curveB):
    curve1 = interpolate_curve_by_x(curveA)
    curve2 = interpolate_curve_by_x(curveB)

    #curve1 = interpolate_curve_by_arclength(curveA)
    #curve2 = interpolate_curve_by_arclength(curveB)

    if len(curve1) < len(curve2):
        longer = curve2
        shorter = curve1
    else:
        longer = curve1
        shorter = curve2

    longer1 = longer[:len(shorter)]
    shorter1 = shorter

    pad_len = len(longer) - len(shorter)
    if pad_len > 0:
        last_point = shorter[-1][None, :]   #seznam ene zadnje točke
        pad = np.repeat(last_point, pad_len, axis=0)    #ponovimo
        shorter2 = pad
        longer2 = longer[len(shorter):]

        diffs = np.vstack([longer1 - shorter1, longer2 - shorter2])
    else:
        diffs = longer1 - shorter1

    dists = np.linalg.norm(diffs, axis=1)
    return np.mean(dists)

In [4]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math

def plot(X_seq, X_sim, j, name):
    """
    Plot actual vs simulated trajectory in 3D with projections.
    Args:
        X_seq (ndarray): ground truth state sequence (T, state_dim)
        X_sim (ndarray): simulated state sequence (T, state_dim)
        j (int): index of jump
        name (str): identifier for saving plots """

    save_dir = f"plots/{name}"
    os.makedirs(save_dir, exist_ok=True)

    
    actual = X_seq[:, :3]   #vzame X,Y,Z iz [x, y, z, vx, vy, vz]
    sim = X_sim[:, :3]          # same iz simulacije

    Xa, Ya, Za = actual[:,0], actual[:,1], actual[:,2]
    Xs, Ys, Zs = sim[:,0], sim[:,1], sim[:,2]

    #napaka = np.mean(np.linalg.norm(actual - sim, axis=1))
    napaka = error_fun(actual, sim)
    length = (Xs[-1]**2 + Ys[-1]**2 + Zs[-1]**2)**(1/2)
    length_a = (Xa[-1]**2 + Ya[-1]**2 + Za[-1]**2)**(1/2)

    fig = plt.figure(figsize=(8, 5))
    ax = fig.add_subplot(111, projection='3d')

    ax.plot3D(Xa, Ya, Za, label="Actual", color="blue")
    ax.plot3D(Xs, Ys, Zs, label="Simulated", color="red", linestyle="--")


    # Ground projections
    ax.plot(Xa, 15, Za, color="blue", alpha=0.3, linestyle=':')
    ax.plot(Xs, 15, Zs, color="red", alpha=0.3, linestyle=':')

    ax.plot(0, Ya, Za, color="blue", alpha=0.3, linestyle=':')
    ax.plot(0, Ys, Zs, color="red", alpha=0.3, linestyle=':')


    ax.set_title(f"Actual vs Simulated Trajectory\n Error: {napaka:.3f}, Simulated length: {length:.1f},  Actual length: {length_a:.1f}")
    ax.set_xlabel("X [m]")
    ax.set_ylabel("Y [m]")
    ax.set_zlabel("Z [m]")
    ax.legend()
    ax.set_box_aspect([1,1,1])
    ax.set_ylim(-15, 15)

    plt.legend()
    #plt.tight_layout()

    filename = f"2SSM flight_simulation{j + 1}.png"
    plt.savefig(os.path.join(save_dir, filename), dpi=300)
    plt.close(fig)
    #plt.show()


In [23]:
#wind_features = ["Speed", "Tangent", "Cross", "Turbulence", 
#   "Speed_quad", "Tangent_quad", "Cross_quad", "Turbulence_quad"]

wind_features = ["Speed", "Tangent", "Cross", "Turbulence"] 

zones = {
    "takeoff": ["W1", "W2", "W3", "W4"],
    "mid":     ["W5", "W6", "W7", "W8"],
    "landing": ["W9", "W10", "W11", "W12"]
}

In [24]:

def preprocess_flight_normalized(df, step=0.05):
    """
    Preprocess a normalized flight dataframe (already interpolated) to generate states (X), observations (Y), and controls (U).
    
    Args:
        df (pd.DataFrame): Normalized flight dataframe (fixed time step).
        step (float): Time step between rows (used for derivatives).
        
    Returns:
        states (np.ndarray): State matrix [time_steps, state_dim].
        observations (np.ndarray): Observation matrix [time_steps, obs_dim].
        controls (np.ndarray): Control matrix [time_steps, control_dim].
    """

    df.columns = df.columns.str.strip()   #izloči imena stolpcev
    df = df.iloc[1:]
    
    df = df.ffill().bfill()   #back fill za manjkajoče vrednosti
    
    x = df["X [m]"].to_numpy()   #save values
    y = df["Y [m]"].to_numpy()
    z = df["Z [m]"].to_numpy()
    
    dt = step
    #vx = df.get("Speed hor. [km/h]", pd.Series([0]*len(x))).to_numpy() * (1000/3600)
    vx = np.gradient(x, dt)
    vy = np.gradient(y, dt)
    vz = np.gradient(z, dt)

    #vz = df.get("Speed vert. [km/h]", pd.Series([0]*len(x))).to_numpy() * (1000/3600)

    speed = df.get("Speed resulting [km/h]", pd.Series([0]*len(x))).to_numpy() * (1000/3600)

    opening = df.get("Opening Angle [°]", pd.Series([0]*len(x))).to_numpy()
    roll_L = df.get("Roll Angle Left [°]", pd.Series([0]*len(x))).to_numpy()
    roll_R = df.get("Roll Angle Right [°]", pd.Series([0]*len(x))).to_numpy()
    yaw_L = df.get("Yaw Angle Left [°]", pd.Series([0]*len(x))).to_numpy()
    yaw_R = df.get("Yaw Angle Right [°]", pd.Series([0]*len(x))).to_numpy()
    stall_L = df.get("Stalling Angle Left [°]", pd.Series([0]*len(x))).to_numpy()
    stall_R = df.get("Stalling Angle Right [°]", pd.Series([0]*len(x))).to_numpy()

    zero = df.get("", pd.Series([0]*len(x))).to_numpy()

    states = np.stack([x, y, z, vx, vy, vz, speed, opening, roll_L, roll_R, yaw_L, yaw_R, stall_L, stall_R], axis=1)   #zgradimo state vector X
    #states = np.stack([x, y, z, speed, opening, roll_L, roll_R, yaw_L, yaw_R, stall_L, stall_R], axis=1)


    height = df.get("Height above ground [m]", pd.Series([0]*len(x))).to_numpy()

    observations = np.stack([x, y, z, height], axis=1)   #zgradimo observation vector Y

    zone_feature_avgs = []
    values = []
    cols = []

    #for feature in wind_features:
    #    for sensor_numb in range(12):
    #        sensor = f"W{sensor_numb + 1}"
    #        cols.append(f"{sensor}_{feature}")
    #        #print(cols)
    #for col in cols:
    #    val = df.get(col, pd.Series([0]*len(x))).to_numpy() 
    #    values.append(val)
    #
    #controls = np.stack(values, axis=1)  # shape: (time_steps, 12)

    #for feature in wind_features:
    #    for zone, sensors in zones.items():
    #        cols = [f"{sensor}_{feature}" for sensor in sensors if f"{sensor}_{feature}" in df.columns]
    #        avg_feature = df[cols].mean(axis=1).to_numpy()
    #        zone_feature_avgs.append(avg_feature)
    #
    #controls = np.stack(zone_feature_avgs, axis=1)  # shape: (time_steps, 12)

    for feature in wind_features:
        for zone, sensors in zones.items():
            cols = [f"{sensor}_{feature}" for sensor in sensors if f"{sensor}_{feature}" in df.columns]
            avg_feature = df[cols].mean(axis=1).to_numpy()
            zone_feature_avgs.append(avg_feature)
            angles = [speed, opening, roll_L, roll_R, yaw_L, yaw_R, stall_L, stall_R]
    for feat in angles:
        zone_feature_avgs.append(feat)
    
    controls = np.stack(zone_feature_avgs, axis=1)
    
    return states, observations, controls


In [25]:
import os
import pandas as pd
import numpy as np

normalized_folder = r'C:\Users\vsi\Desktop\ijs\smucarski_skoki\project\simulation\2024_03_Planica_12_winds\cleaned\cleaned_quad\normalized_quad'
combined_output = os.path.join(normalized_folder, "combined_dataset.npz")
file_names = []

X_list, Y_list, U_list = [], [], []

for filename in os.listdir(normalized_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(normalized_folder, filename)
        df = pd.read_csv(file_path)

        X_state, Y_obs, U_ctrl = preprocess_flight_normalized(df)

        X_list.append(X_state)
        Y_list.append(Y_obs)
        U_list.append(U_ctrl)
        file_names.append(filename)

        #if np.any(np.isnan(X_state)):
        #    print(f"NaN values in {filename}")

        print(f"Processed {filename} -> X:{X_state.shape}, Y:{Y_obs.shape}, U:{U_ctrl.shape}")

# Save combined dataset
#np.savez(combined_output, X=np.array(X_list, dtype=object), Y=np.array(Y_list, dtype=object), U=np.array(U_list, dtype=object))
print(f"\nCombined dataset saved to {combined_output}")


Processed 999_999_Jumper_Anon_ANO_1_20240409-102625_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(133, 14), Y:(133, 4), U:(133, 20)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102742_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(129, 14), Y:(129, 4), U:(129, 20)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102831_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(142, 14), Y:(142, 4), U:(142, 20)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102832_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(135, 14), Y:(135, 4), U:(135, 20)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102833_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(132, 14), Y:(132, 4), U:(132, 20)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102834_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(131, 14), Y:(131, 4), U:(131, 20)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102835_C_OfficialResults_cleaned_quad-cleaned_nor

In [34]:
len(X_list)

202

In [36]:
weird_vrednosti = X_list.pop(169), Y_list.pop(169), U_list.pop(169)
weird_vrednosti

(array([[ 2.60769231e+00,  3.23076923e-02, -4.50000000e-01, ...,
         -2.56438462e+00, -9.24676923e+00, -9.01207692e+00],
        [ 3.92000000e+00,  4.00000000e-02, -6.20000000e-01, ...,
         -3.74950000e+00, -2.85500000e+00, -5.64800000e+00],
        [ 5.25315789e+00,  4.78947368e-02, -8.16315789e-01, ...,
         -5.59552632e+00,  2.99442105e+00, -2.81778947e+00],
        ...,
        [ 1.90664286e+02, -7.98571429e-01, -1.14065714e+02, ...,
         -1.14168571e+01, -7.54939286e+00, -1.23167500e+01],
        [ 1.92083333e+02, -6.85000000e-01, -1.15143333e+02, ...,
         -1.00856667e+01, -7.13716667e+00, -1.63811667e+01],
        [ 1.93372000e+02, -6.50000000e-01, -1.16170000e+02, ...,
         -7.62890000e+00, -1.34907000e+01, -2.44457000e+01]],
       shape=(147, 14)),
 array([[ 2.60769231e+00,  3.23076923e-02, -4.50000000e-01,
          2.81461538e+00],
        [ 3.92000000e+00,  4.00000000e-02, -6.20000000e-01,
          2.85000000e+00],
        [ 5.25315789e+00,  4.78

In [37]:
len(X_list)

201

In [27]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge

def fit_ssm_cv_quad(X_list, U_list, Y_list, n_splits=5, alpha=1e-3):
    """
    Fit an SSM model using K-fold cross-validation across jumps.

    Args:
        X_list, U_list, Y_list: lists of arrays (one per jump)
        n_splits (int): number of folds (or = len(X_list) for LOOCV)
        alpha (float): ridge regularization parameter

    Returns:
        avg_train_err, avg_test_err, models (list of (A,B,C,D) per fold)
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    train_errors, test_errors = [], []
    models = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_list)):
        X_t = np.vstack([seq[:-1] for i, seq in enumerate(X_list) if i in train_idx])
        X_next = np.vstack([seq[1:] for i, seq in enumerate(X_list) if i in train_idx])
        U_t = np.vstack([u[:-1] for i, u in enumerate(U_list) if i in train_idx])
        Y_t = np.vstack([y[:-1] for i, y in enumerate(Y_list) if i in train_idx])

        # Regression [X_next] ~ [X_t | U_t]
        Phi = np.hstack([X_t, U_t])
        ridge = Ridge(alpha=alpha, fit_intercept=False)     #ridge is just MNK z regularizacijo
        ridge.fit(Phi, X_next)
        Theta = ridge.coef_.T

        state_dim = X_t.shape[1]
        #control_dim = U_t.shape[1]
        A = Theta[:state_dim, :].T
        B = Theta[state_dim:, :].T

        # Regression [Y_t] ~ [X_t | U_t]
        Phi_y = np.hstack([X_t, U_t])
        ridge_y = Ridge(alpha=alpha, fit_intercept=False)
        ridge_y.fit(Phi_y, Y_t)
        Theta_y = ridge_y.coef_.T

        obs_dim = Y_t.shape[1]
        C = Theta_y[:state_dim, :].T
        D = Theta_y[state_dim:, :].T

        models.append((A, B, C, D))

        train_errs = []
        for i in train_idx:  #should be one, but just in case
            x0 = X_list[i][0]
            U_seq = U_list[i]
            X_seq = X_list[i]
            
            X_sim = [x0]
            for t in range(len(U_seq)-1):
                x_next = A @ X_sim[-1] + B @ U_seq[t]
                X_sim.append(x_next)
            X_sim = np.array(X_sim)
        
            train_err = error_fun(X_seq[:, :3], X_sim[:, :3])
            train_errs.append(train_err)

        train_errors.append(np.mean(train_errs))

        # --- compute test error ---
        test_errs = []
        for j in test_idx:
            x0 = X_list[j][0]      
            U_seq = U_list[j]
            X_seq = X_list[j]

            # simulate with learned model
            X_sim = [x0]
            for t in range(len(U_seq)-1):
                x_next = A @ X_sim[-1] + B @ U_seq[t]
                X_sim.append(x_next)
            X_sim = np.array(X_sim)
            
            test_err = error_fun(X_seq[:, :3], X_sim[:, :3])
            test_errs.append(test_err)

            plot(X_seq[:, :3], X_sim[:, :3], j, "15-S3")
        test_errors.append(np.mean(test_errs))

        #print(train_err, np.mean(test_errs))
        print(np.mean(train_errs), np.mean(test_errs))


    return np.mean(train_errors), np.mean(test_errors), models


In [38]:
avg_train_err, avg_test_err, models = fit_ssm_cv_quad(X_list, U_list, Y_list, n_splits=len(X_list), alpha=10)

print("average train error:", avg_train_err)
print("average test error:", avg_test_err)

1.3229782785053408 2.0868181741423992
1.323166553971516 0.8708388758144427
1.3226709890743127 0.7951590577684633
1.3199868104251835 1.6455126084746645
1.3204151821439358 1.2982478478490842
1.3251292893231146 0.48485955379098117
1.322557823198278 0.9898882086329363
1.3238164044589977 0.5593911378839266
1.3138502133912684 2.0468461142846883
1.3138359512922981 1.9779044841540807
1.322723785892622 0.7586104464049652
1.323819559045105 1.048253089186798
1.320786553329774 1.4566761957074703
1.3212138898162311 1.069410180617719
1.3199605480949077 1.3830479922445296
1.323693410185471 1.0635267034098068
1.3177818943230375 2.0783790798204707
1.322735048900245 1.098396110299259
1.322688093771593 0.9536075770190404
1.3212634626336797 1.069648599202376
1.3212712314754274 1.4597738554996378
1.317502565298344 2.1267293362747792
1.3246588742779328 0.48962805299095874
1.323313480522475 0.770955818016336
1.3172037520855673 1.5220978580884539
1.3242749669390326 0.4173471218664386
1.3227837444233737 0.7500